# Two-Column Grid Navigation — Reproducible Analysis (from raw data)

Builds the full analysis from scratch, starting from the raw Qilin dataset on
HuggingFace

In [ ]:
%pip install datasets statsmodels pandas numpy scipy matplotlib
print("\nDone. Restart the kernel if prompted, then Run All.")

In [ ]:
import os, re
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
import statsmodels.api as sm
import matplotlib.pyplot as plt

# ---- build controls ----
REBUILD_FROM_RAW = True          # True = build combined log from raw Qilin; False = reuse cached CSV
COMBINED_CSV     = "qilin_combined_sessions.csv"   # cache written/read here
FLAG_CACHE       = "qilin_notes_flags.csv"
TRAJ_CSV         = "qilin_spatial_trajectories.csv"  # Figure 1 only (descriptive)

# raw Qilin splits to use (the canonical recipe described in the thesis)
SEARCH_SPLITS = ["search_train", "search_test"]
REC_SPLITS    = ["recommendation_train", "recommendation_test"]
MOBILE        = ["iOS", "Android"]   # mobile platforms kept in BOTH modules

# ---- column names ----
COL_USER, COL_SESSION, COL_REQUEST = "user_idx", "session_idx", "search_idx"
COL_INTENT = "session_intent"
GOAL_VALUE = "goal-oriented"
SEARCH_DETAILS = "search_result_details_with_idx"
REC_DETAILS    = "rec_result_details_with_idx"

# ---- analysis constants ----
SHALLOW_MAX, MID_MAX = 4, 9
AD_POSITIONS, PRE_AD_POSITIONS = (3, 6), (2, 5)
PLOT_DEPTH_CAP = 20
COMMERCIAL_FLAG_VALUE = 2.0
print("config loaded")

In [ ]:
def clicked_positions(row):
    text = str(row.get(SEARCH_DETAILS, ""))
    if text in ("nan", "None", "<NA>") or not text.strip():
        text = str(row.get(REC_DETAILS, ""))
    if text in ("nan", "None", "<NA>") or not text.strip():
        return []
    out = []
    for block in re.findall(r"\{[^{}]+\}", text):
        c = re.search(r"['\"]click['\"]:\s*(\d+)", block)
        p = re.search(r"['\"]position['\"]:\s*(\d+)", block)
        if c and p and int(c.group(1)) == 1:
            out.append(int(p.group(1)))
    return out

def engagement(positions):
    sides = {"Left" if p % 2 else "Right" for p in positions}
    return "Cross-Column" if len(sides) == 2 else "Single-Column"

def exposed_positions_notes(row):
    text = str(row.get(SEARCH_DETAILS, ""))
    if text in ("nan", "None", "<NA>") or not text.strip():
        text = str(row.get(REC_DETAILS, ""))
    if text in ("nan", "None", "<NA>") or not text.strip():
        return []
    out = []
    for block in re.findall(r"\{[^{}]+\}", text):
        p = re.search(r"['\"]position['\"]:\s*(\d+)", block)
        n = re.search(r"['\"]note_idx['\"]:\s*(\d+)", block)
        if p and n:
            out.append((int(p.group(1)), int(n.group(1))))
    return out

def cramers_v(table):
    table = np.asarray(table)
    chi2 = chi2_contingency(table)[0]
    n = table.sum(); r, c = table.shape
    return float(np.sqrt((chi2 / n) / min(r - 1, c - 1)))

print("functions defined")

In [ ]:
def load_splits(config_list):
    from datasets import load_dataset
    frames = []
    for cfg in config_list:
        try:
            d = load_dataset("THUIR/Qilin", cfg, split="train").to_pandas()
            print(f"  loaded {cfg}: {len(d):,} rows")
            frames.append(d)
        except Exception as e:
            print(f"  WARNING could not load {cfg}: {repr(e)[:90]}")
    if not frames:
        raise RuntimeError("no splits loaded")
    return pd.concat(frames, ignore_index=True)

if REBUILD_FROM_RAW or not os.path.exists(COMBINED_CSV):
    from datasets import load_dataset
    print("=== building combined log from raw Qilin ===")

    print("user features:")
    ufe = load_dataset("THUIR/Qilin", "user_feat", split="train").to_pandas()
    print(f"  loaded user_feat: {len(ufe):,} rows")

    print("\nSEARCH module (goal-oriented):")
    search = load_splits(SEARCH_SPLITS)
    search = search.merge(ufe, on=COL_USER, how="left")
    n_before = len(search)
    search = search[search["platform"].isin(MOBILE)]
    print(f"  search after user merge:      {n_before:,}")
    print(f"  search after MOBILE filter:   {len(search):,}")

    print("\nRECOMMENDATION module (exploratory):")
    rec = load_splits(REC_SPLITS)
    rec = rec.merge(ufe, on=COL_USER, how="left")
    n_before = len(rec)
    rec = rec[rec["platform"].isin(MOBILE)]            # the filter applied to BOTH modules
    print(f"  rec after user merge:         {n_before:,}")
    print(f"  rec after MOBILE filter:      {len(rec):,}")

    search["module"] = "Search"
    rec["module"]    = "Recommendation"
    df = pd.concat([search, rec], ignore_index=True)
    df[COL_INTENT] = df["module"].map({"Search": "goal-oriented",
                                       "Recommendation": "exploratory"})
    df.to_csv(COMBINED_CSV, index=False)
    print(f"\nCOMBINED (both modules, mobile-only): {len(df):,} rows  ->  cached to {COMBINED_CSV}")
else:
    df = pd.read_csv(COMBINED_CSV)
    print(f"reused cached {COMBINED_CSV}: {len(df):,} rows")

In [ ]:
df["positions"] = df.apply(clicked_positions, axis=1)
df["n_clicks"]  = df["positions"].apply(len)

df = df.sort_values([COL_USER, COL_SESSION, COL_REQUEST]).reset_index(drop=True)
df["row_id"] = np.arange(len(df))
df["depth"]  = df.groupby([COL_USER, COL_SESSION]).cumcount() + 1

multi = df[df["n_clicks"] >= 2].copy()
multi["engagement"] = multi["positions"].apply(engagement)
multi["cross"]      = (multi["engagement"] == "Cross-Column").astype(int)
multi["stage"] = np.select(
    [multi["depth"] <= SHALLOW_MAX,
     (multi["depth"] > SHALLOW_MAX) & (multi["depth"] <= MID_MAX),
     multi["depth"] > MID_MAX],
    ["1. Shallow", "2. Mid", "3. Deep"], default="?")
multi["pre_ad"] = multi["positions"].apply(lambda ps: any(p in PRE_AD_POSITIONS for p in ps))
multi["zone"]   = np.where(multi["pre_ad"], "Pre-Ad Zone", "Organic Zone")

assert multi.duplicated("row_id").sum() == 0, "DUPLICATE ROWS — fan-out happened"
print("multi-click requests:", f"{len(multi):,}")
print("band counts:", multi["stage"].value_counts().to_dict())

In [ ]:
print("=== counts after both-module mobile filter ===")
print(f"requests:              {len(df):,}")
print(f"users:                 {df[COL_USER].nunique():,}")
print(f"sessions:              {df[COL_SESSION].nunique():,}")
print(f"multi-click requests:  {len(multi):,}")
if "platform" in df.columns:
    print("\nplatform mix:")
    print((df["platform"].value_counts(normalize=True) * 100).round(1).to_string())
print("\nby module:")
for m, g in df.groupby("module"):
    print(f"  {m}: requests={len(g):,}  users={g[COL_USER].nunique():,}  sessions={g[COL_SESSION].nunique():,}")

In [ ]:
def parse_click_timestamps(row):
    text = str(row.get(SEARCH_DETAILS, ""))
    if text in ("nan", "None", "<NA>") or not text.strip():
        text = str(row.get(REC_DETAILS, ""))
    if text in ("nan", "None", "<NA>") or not text.strip():
        return []
    ts = []
    for block in re.findall(r"\{[^{}]+\}", text):
        c = re.search(r"['\"]click['\"]:\s*(\d+)", block)
        t = re.search(r"['\"](?:search|request)_timestamp['\"]:\s*([\d.]+)", block)
        if c and t and int(c.group(1)) == 1:
            ts.append(float(t.group(1)))
    return ts

mc = multi.copy()
mc["click_ts"] = mc.apply(parse_click_timestamps, axis=1)
print("share of clicks in multi-click requests:",
      round(100 * multi["n_clicks"].sum() / df["n_clicks"].sum(), 1), "%")

def ordering_quality(ts):
    if len(ts) < 2:  return "single-or-none"
    u = len(set(ts))
    if u == 1:       return "all-tied (single timestamp)"
    if u == len(ts): return "fully-ordered"
    return "partial-ties"

mc["ord_q"] = mc["click_ts"].apply(ordering_quality)
orderable = mc[mc["ord_q"] != "single-or-none"]
print("\nordering quality (requests with >=2 timestamps):")
print((orderable["ord_q"].value_counts(normalize=True) * 100).round(0).astype(int).to_string())

In [ ]:
baseline = (multi["engagement"].value_counts(normalize=True) * 100).round(1)
print(baseline.to_string())

In [ ]:
it = pd.crosstab(multi[COL_INTENT], multi["engagement"])
print((it.div(it.sum(axis=1), axis=0) * 100).round(1).to_string())
print("\nCramer's V =", round(cramers_v(it.values), 3))

In [ ]:
st = pd.crosstab(multi["stage"], multi["engagement"])
st_pct = (st.div(st.sum(axis=1), axis=0) * 100).round(1)
print(st_pct.to_string())
print("\nCramer's V =", round(cramers_v(st.values), 3))

In [ ]:
if os.path.exists(FLAG_CACHE):
    notes = pd.read_csv(FLAG_CACHE)
    print("loaded cached notes flags")
else:
    from datasets import load_dataset
    print("downloading notes from HuggingFace (one time)...")
    notes = load_dataset("THUIR/Qilin", "notes", split="train").to_pandas()[["note_idx", "commercial_flag"]]
    notes.to_csv(FLAG_CACHE, index=False)
    print("cached to", FLAG_CACHE)
flag = dict(zip(notes["note_idx"], notes["commercial_flag"]))
print("flag map built:", len(flag), "notes")

In [ ]:
from collections import defaultdict
pos_tot, pos_comm = defaultdict(int), defaultdict(int)
for _, row in df.iterrows():
    for pos, nid in exposed_positions_notes(row):
        f = flag.get(nid)
        if f is None or f != f:
            continue
        pos_tot[pos] += 1
        if f == COMMERCIAL_FLAG_VALUE:
            pos_comm[pos] += 1
print("position : commercial-exposure rate | n")
for p in sorted(pos_tot):
    if p <= 8 and pos_tot[p] > 200:
        print(f"{p:>3}      : {100*pos_comm[p]/pos_tot[p]:5.1f}% | {pos_tot[p]}")

In [ ]:
zn = pd.crosstab(multi["zone"], multi["engagement"])
zn_pct = (zn.div(zn.sum(axis=1), axis=0) * 100).round(1)
zn_pct["Mean Clicks"] = multi.groupby("zone")["n_clicks"].mean().round(2)
print("TABLE II — position-based zone (uncontrolled):")
print(zn_pct.to_string())
print("\nCramer's V =", round(cramers_v(zn.values), 3))

print("\nTABLE III — cross-column % by zone, within click-count strata:")
rows = []
for k in (2, 3, 4):
    sub = multi[multi["n_clicks"] == k]
    g = sub.groupby("zone")["cross"].mean() * 100
    rows.append((str(k), g.get("Organic Zone", np.nan), g.get("Pre-Ad Zone", np.nan)))
sub5 = multi[multi["n_clicks"] >= 5]
g = sub5.groupby("zone")["cross"].mean() * 100
rows.append(("5+", g.get("Organic Zone", np.nan), g.get("Pre-Ad Zone", np.nan)))
strata = pd.DataFrame(rows, columns=["Clicks/Request", "Organic zone", "Pre-Ad zone"])
strata["Difference"] = (strata["Pre-Ad zone"] - strata["Organic zone"]).round(1)
print(strata.round(1).to_string(index=False))

In [ ]:
def commercial_any(row):
    return int(any(flag.get(nid) == COMMERCIAL_FLAG_VALUE
                   for _, nid in exposed_positions_notes(row)))
def commercial_early(row):
    return int(any(flag.get(nid) == COMMERCIAL_FLAG_VALUE and pos <= 6
                   for pos, nid in exposed_positions_notes(row)))
multi["comm_any"]   = multi.apply(commercial_any,   axis=1)
multi["comm_early"] = multi.apply(commercial_early, axis=1)
print("share exposing a commercial note:", round(multi["comm_any"].mean(), 3))
print("share exposing one early (<=6):  ", round(multi["comm_early"].mean(), 3))

In [ ]:
d = multi.copy()
d["clicks_c"]    = d["n_clicks"] - d["n_clicks"].mean()
d["intent_goal"] = (d[COL_INTENT] == GOAL_VALUE).astype(int)
d["stage_mid"]   = (d["stage"] == "2. Mid").astype(int)
d["stage_deep"]  = (d["stage"] == "3. Deep").astype(int)
d["y"]           = d["cross"]

X = sm.add_constant(d[["clicks_c", "intent_goal", "stage_mid", "stage_deep", "comm_any"]])
m = sm.Logit(d["y"], X).fit(disp=0, cov_type="HC3")
out = np.exp(m.params).rename("OR").to_frame()
out[["CI_low", "CI_high"]] = np.exp(m.conf_int()).values
out["p"] = m.pvalues
out = out.rename(index={"const": "Intercept", "clicks_c": "Clicks per request (centered)",
    "intent_goal": "Intent: goal-oriented", "stage_mid": "Session depth: Mid",
    "stage_deep": "Session depth: Deep", "comm_any": "Ad exposed (commercial)"})
print("Reference: intent=exploratory, depth=Shallow, ad=not exposed. HC3 SEs.")
print(out.round(3).to_string())

In [ ]:
def label(v): return "Negligible" if v < 0.10 else ("Modest" if v < 0.30 else "Large")
summary = pd.DataFrame([
    {"SQ":"SQ1","Variable":"User Intent","V":round(cramers_v(it.values),3),"Effect":label(cramers_v(it.values))},
    {"SQ":"SQ2","Variable":"Session Depth","V":round(cramers_v(st.values),3),"Effect":label(cramers_v(st.values))},
    {"SQ":"SQ3","Variable":"Ad Proximity (position-based)","V":round(cramers_v(zn.values),3),"Effect":"Artifactual (see exposure model)"},
])
print(summary.to_string(index=False))

Figures

In [ ]:
# Figure 1 — session-depth distribution (descriptive, PER-INTERACTION).
if os.path.exists(TRAJ_CSV):
    traj = pd.read_csv(TRAJ_CSV)
    active = traj[traj["spatial_transition"] != "Initial Interaction"].copy()
    active["cdepth"] = active.groupby([COL_USER, COL_SESSION]).cumcount() + 1
    counts = (active[active["cdepth"] <= PLOT_DEPTH_CAP]["cdepth"]
              .value_counts().sort_index()
              .reindex(range(1, PLOT_DEPTH_CAP + 1), fill_value=0))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(counts.index, counts.values, width=0.85,
           color="#4C72B0", edgecolor="white", linewidth=0.5)
    ax.axvline(SHALLOW_MAX + 0.5, color="#C44E52", linestyle="--", linewidth=1.2,
               label="End of Shallow band")
    ax.axvline(MID_MAX + 0.5, color="#8172B3", linestyle="-", linewidth=1.2,
               label="End of Mid band")
    ax.set_title("Distribution of Session Depth")
    ax.set_xlabel("Session Depth (Nth interaction)")
    ax.set_ylabel("Frequency (Number of interactions)")
    ax.set_xticks(range(1, PLOT_DEPTH_CAP + 1))
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig("SQ2_Depth_Distribution.png", dpi=200)
    plt.show()
    print("saved SQ2_Depth_Distribution.png")
else:
    print(f"{TRAJ_CSV} not found \u2014 skipping Figure 1 (descriptive only; "
          "place the trajectory file in this folder to regenerate it). "
          "All other results still run.")

In [ ]:
# Figure 2 — engagement across depth bands
labels = ["Shallow (1-4 requests)", "Mid (5-9 requests)", "Deep (10+ requests)"]
order  = ["1. Shallow", "2. Mid", "3. Deep"]
cross  = [st_pct.loc[s, "Cross-Column"] for s in order]
single = [st_pct.loc[s, "Single-Column"] for s in order]
x = np.arange(len(labels)); w = 0.38
fig, ax = plt.subplots(figsize=(8,5))
b1 = ax.bar(x-w/2, cross, w, label="Cross-Column", color="#4C72B0")
b2 = ax.bar(x+w/2, single, w, label="Single-Column", color="#DD8452")
for b in (b1,b2): ax.bar_label(b, fmt="%.1f%%", padding=2, fontsize=9)
ax.set_title("Column Engagement Across Session Depth"); ax.set_ylabel("Percentage (%)")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0,100); ax.legend()
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("session_depth_bar.png", dpi=200); plt.show()

In [ ]:
# Figure 3 — forest plot
terms = ["Clicks per request (centered)","Intent: goal-oriented","Session depth: Mid","Session depth: Deep","Ad exposed (commercial)"]
ors = out.loc[terms,"OR"].values; lo = out.loc[terms,"CI_low"].values; hi = out.loc[terms,"CI_high"].values
yp = np.arange(len(terms))[::-1]
fig, ax = plt.subplots(figsize=(8,5))
ax.errorbar(ors, yp, xerr=[ors-lo, hi-ors], fmt="o", color="#4C72B0", capsize=4, linewidth=1.5, markersize=7)
ax.axvline(1.0, color="grey", linestyle="--", linewidth=1)
ax.set_yticks(yp); ax.set_yticklabels(terms); ax.set_xlabel("Odds Ratio (OR)")
ax.set_title("Logistic Regression: Odds of Cross-Column Engagement")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig("odds_ratios_forest.png", dpi=200); plt.show()

In [ ]:
Appendix

In [ ]:
cols = [c for c in [COL_USER, COL_SESSION, COL_REQUEST, "query_from_type", COL_INTENT] if c in df.columns]
print(df[cols].head(8).to_string(index=False))